v3:
1. Change to Monthly Cadence (starting 2026)
2. Add 'Month' column (ISO YYYY-MM format) to all data pipelines
3. Pre-2026 data remains quarterly (Month = None)
4. 2026+ data: monthly aggregation with forward-fill for survey scores
5. Dynamic Quarter/Month derivation (no hardcoded CASE statements for 2026+)

# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime
import glob
import os
import re

# Helper Functions - Monthly/Quarterly Logic

In [2]:
# ---- MONTHLY CADENCE CUTOVER DATE ----
MONTHLY_START_DATE = '2026-01-01'

def derive_month_from_date(date_series):
    """Derive Month (YYYY-MM) from a date column. Returns None for pre-2026 dates."""
    date_series = pd.to_datetime(date_series)
    return np.where(
        date_series >= pd.Timestamp(MONTHLY_START_DATE),
        date_series.dt.strftime('%Y-%m'),
        None
    )

def derive_quarter_from_date(date_series):
    """Derive Quarter string (e.g., Q1'26) from a date column."""
    date_series = pd.to_datetime(date_series)
    return date_series.apply(
        lambda d: f"Q{d.quarter}'{str(d.year)[2:]}" if pd.notna(d) else None
    )

def derive_quarter_from_month(month_series):
    """Derive Quarter string from Month (YYYY-MM) column."""
    def _to_quarter(m):
        if pd.isna(m) or m is None:
            return None
        year = m[:4]
        month_num = int(m[5:7])
        q = (month_num - 1) // 3 + 1
        return f"Q{q}'{year[2:]}"
    return month_series.apply(_to_quarter)

def get_groupby_keys(include_month=True):
    """Return standard groupby keys. Include Month for 2026+ data."""
    keys = ['Location', 'JobFamilyGroup', 'Quarter']
    if include_month:
        keys.append('Month')
    return keys

def get_merge_keys(include_month=True):
    """Return standard merge keys. Include Month for 2026+ data."""
    keys = ['Location', 'JobFamilyGroup', 'Quarter']
    if include_month:
        keys.append('Month')
    return keys

def forward_fill_scores(df, score_cols=['Score'], group_cols=['Location', 'JobFamilyGroup']):
    """Forward-fill survey scores for months within a quarter where no survey was conducted.
    Only applies to 2026+ monthly data."""
    df = df.sort_values(group_cols + ['Month'])
    for col in score_cols:
        df[col] = df.groupby(group_cols)[col].ffill()
    return df

def parse_period_from_filename(filename):
    """Parse Quarter or Month from filename.
    Handles both quarterly (Q1'25) and monthly (2026-01) formats."""
    basename = filename.split('_')[-1].split('.')[0]
    # Check if it's ISO month format (YYYY-MM)
    if re.match(r'\d{4}-\d{2}', basename):
        return {'Month': basename, 'Quarter': derive_quarter_from_month(pd.Series([basename])).iloc[0]}
    else:
        # Quarterly format like Q1'25
        return {'Month': None, 'Quarter': basename}

# Snowflake SQL snippets for dynamic Month/Quarter derivation
SNOWFLAKE_MONTH_QUARTER_SQL = """
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter
"""

print('Helper functions loaded. Monthly cadence starts from:', MONTHLY_START_DATE)

Helper functions loaded. Monthly cadence starts from: 2026-01-01


# Connections

In [3]:
from carvana_sql import SQLServer # type: ignore
db_conn = SQLServer.connect('PEOPLEOPSPROD,1439')

In [4]:
cnx_str = "Driver={ODBC Driver 17 for SQL Server};Server=PEOPLEOPSPROD,1439;Database=PEOPLEOPS;Trusted_Connection=yes;"
cnx_str_app = "Driver={ODBC Driver 17 for SQL Server};Server=dwhcar-l;Database=OpsGroup;Trusted_Connection=yes;"

import pyodbc
odbcconn = pyodbc.connect(cnx_str)
odbcconnops = pyodbc.connect(cnx_str_app)
odbcquery = 'Select Top 10 * from PeopleOps.tenstreet.FactDriver'
odbcdf = pd.read_sql_query(odbcquery, odbcconn)

C:\Users\E102068\AppData\Local\Temp\ipykernel_16560\2098714686.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  odbcdf = pd.read_sql_query(odbcquery, odbcconn)


In [5]:
# Today's date
date_today=pd.to_datetime(datetime.date.today())

In [6]:
from carvana_sql import Snowflake

SnowflakeWarehouse = 'PEOPLEOPS_ANALYTICS'
SnowflakeRole = 'PEOPLEOPS_ANALYTICS'
SnowflakeEmail = 'anish.middela@carvana.com'
SnowflakeDatabase = "PEOPLEOPS_SANDBOX"
Snowflakeschema="SITERISK"

db_snowflake = Snowflake.connect(
                username=SnowflakeEmail,                
                role=SnowflakeRole,
                warehouse=SnowflakeWarehouse)

db_snowflake_write = Snowflake.connect(
                username=SnowflakeEmail,                
                role=SnowflakeRole,
                warehouse=SnowflakeWarehouse,
                database=SnowflakeDatabase,
                schema= Snowflakeschema )

# CheckPoint

### Location Files

In [7]:
# Netwrok csv pull
network_location_path="X:\People_Operations\Secured\Talent Analytics\Anish Middela\SiteRisk"

# Level 1 csv files
network_location_files= glob.glob(os.path.join(network_location_path, "**\*.csv"),recursive=True)

In [8]:
network_location_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q3_2025.csv',
 'X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q4_2025.csv',
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\ADESA\\glint_Q1'24_ADESA.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Logistics.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Market Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Post Production.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Reconditioning.csv",
 "X:\\People_Operations\\Secured\\Talent Analyt

In [9]:
glint_files = []
injury_files = []
navex_files = []
overall_files=[]
checkpoint_files=[]


for path in network_location_files:
    if "Overall" in path:
        overall_files.append(path)
    elif "Navex" in path:
        navex_files.append(path)
    elif "Checkpoint" in path:
        checkpoint_files.append(path)
    elif "Glint" in path:
        glint_files.append(path)
    elif "Injuries" in path:
        injury_files.append(path)

In [10]:
# files=os.listdir("C:/Users/E102068/Desktop/SiteRisk Dashboard/Glint Reports/Q1_24")

# # Files with overall data
# main_path="Glint Reports\Q1_24\Overall"

# main_files= glob.glob(main_path+"\*.csv")

In [11]:
navex_files

["X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q1'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q2'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q3'25.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Navex\\navex_Q4'25.csv"]

In [12]:
glint_files

["X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\ADESA\\glint_Q1'24_ADESA.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Logistics.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Market Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Post Production.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Reconditioning.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carvana_Q1'24_Regulatory Operations.csv",
 "X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Glint Reports\\Q1'24\\Carvana\\glint_Carva

In [ ]:
location_df = pd.DataFrame()
# Appending all the csv files into one df
for file in glint_files:
    df_temp=pd.read_csv(file)
    df_temp['JobFamilyGroup']= file.split('_')[-1].split('.')[0]
    
    # v3: Quarter comes from folder path (same as v2), Month = None (Glint is quarterly)
    df_temp['Quarter'] = file.split('_')[-2]
    df_temp['Month'] = None

    location_df=pd.concat([location_df,df_temp],ignore_index=True)

In [14]:
# Select only neccesary columns
location_df=location_df.iloc[:,[0,5,6,15,16]]

# Rename
location_df= location_df.rename(columns={location_df.columns[1]:'Score',location_df.columns[2]:'eSat Change'})

# Filter out Location ==ALL
location_df = location_df[~(location_df['Location'] == 'All')]

In [15]:
# Convert 'Score' column to numeric, errors='coerce' will replace non-numeric values with NaN
location_df['Score'] = pd.to_numeric(location_df['Score'], errors='coerce')
location_df['eSat Change'] = pd.to_numeric(location_df['eSat Change'], errors='coerce')

# Fill na ==0
location_df=location_df.fillna(0)

# Peakon / Checkpoint

In [17]:
checkpoint_files

['X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q3_2025.csv',
 'X:\\People_Operations\\Secured\\Talent Analytics\\Anish Middela\\SiteRisk\\Checkpoint\\Checkpoint_Q4_2025.csv']

In [ ]:
# Checkpoint Data Pull
# v3: Filename format is Checkpoint_Q4_2025.csv -> parse Q4 and 2025 to build Q4'25
checkpoint_df = pd.DataFrame()
for file in checkpoint_files:
    df = pd.read_csv(file)
    # Parse Quarter from filename: Checkpoint_Q4_2025.csv -> Q4'25
    parts = os.path.basename(file).replace('.csv','').split('_')
    quarter_num = parts[-2]  # e.g., 'Q4'
    year = parts[-1]         # e.g., '2025'
    df['Quarter'] = f"{quarter_num}'{year[2:]}"  # e.g., "Q4'25"
    df['Month'] = None  # Checkpoint stays quarterly
    checkpoint_df = pd.concat([checkpoint_df, df], ignore_index=True)

Peakon Data Cleaning

In [ ]:
checkpoint_df.columns

In [ ]:
checkpoint_df['JobFamilyGroup'] = np.where(checkpoint_df['JobFamilyGroup'].str.contains('ADESA'), 'ADESA',checkpoint_df['JobFamilyGroup'])

In [ ]:
checkpoint_df['Employee_Size_Checkpoint'] = checkpoint_df['Participants']/checkpoint_df['Participation_Percentage']
checkpoint_df['Employee_Size_Checkpoint'] = checkpoint_df['Employee_Size_Checkpoint'].round(0).astype(int)

In [ ]:
checkpoint_df = checkpoint_df[['Location','JobFamilyGroup','Participants','Employee_Size_Checkpoint','Score','Quarter','Month']]

In [ ]:
# v3: Checkpoint stays quarterly - group by Location/JobFamilyGroup/Quarter only (no Month)
# Scores will be forward-filled into monthly rows later during the main_df merge
checkpoint_df = checkpoint_df.groupby(['Location','JobFamilyGroup','Quarter']).agg({'Participants':'sum','Employee_Size_Checkpoint':'sum','Score':'mean'}).reset_index()
checkpoint_df['Month'] = None  # Checkpoint is quarterly, Month stays None

In [ ]:
checkpoint_df['Score'] = checkpoint_df['Score'].round(1)

In [ ]:
## Trim whitespace from Location, JobFamilyGroup, and Quarter columns
checkpoint_df['Location'] = checkpoint_df['Location'].str.strip()
checkpoint_df['JobFamilyGroup'] = checkpoint_df['JobFamilyGroup'].str.strip()
checkpoint_df['Quarter'] = checkpoint_df['Quarter'].str.strip()

In [ ]:
# Create empty columns in location_df for merging later
location_df['Participants']=None
location_df['Employee_Size_Checkpoint']=None

In [ ]:
# v3: Include Month column
if 'Month' not in location_df.columns:
    location_df['Month'] = None
location_df=location_df[['Location','JobFamilyGroup','Participants','Employee_Size_Checkpoint','Score','Quarter','Month']]

In [ ]:
location_df = pd.concat([location_df, checkpoint_df], ignore_index=True)

In [ ]:
location_df['Quarter'].value_counts()

### Necessary JobFamilyGroups

In [ ]:
necessary_JobFamilyGroups = ['Logistics','Market Operations','Post Production',
                             'Reconditioning','Regulatory Operations','Wholesale Operations','ADESA','Registration']

In [ ]:
location_df = location_df[location_df['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [ ]:
# necessary_JobFamilyGroups= location_df['JobFamilyGroup'].unique().tolist()

In [ ]:
# Get value counts for 'JobFamilyGroup' in Q4'25 quarter
location_df[location_df['Quarter'] == "Q4'25"]['JobFamilyGroup'].value_counts()

In [ ]:
# Compare value counts for each quarter side by side
# This will create a DataFrame with JobFamilyGroup as rows and Quarters as columns
value_counts_df = location_df.groupby(['Quarter', 'JobFamilyGroup']).size().unstack(fill_value=0)
value_counts_df

In [ ]:
location_df['JobFamilyGroup'] = np.where(location_df['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', location_df['JobFamilyGroup'])

# v3: Forward-fill checkpoint/survey scores across months
# Logic: Quarterly checkpoint data (Month=None) gets forward-filled into monthly rows.
# Example: Q4'25 score fills into 2026-01, 2026-02, 2026-03 until Q1'26 data overwrites it.
# Sort by Quarter then Month so quarterly (None) rows come before monthly rows
location_df = location_df.sort_values(['Location', 'JobFamilyGroup', 'Quarter', 'Month'])
for col in ['Score', 'Participants', 'Employee_Size_Checkpoint']:
    location_df[col] = location_df.groupby(['Location', 'JobFamilyGroup'])[col].ffill()

# Injuries

In [ ]:
# Old Logic
# injury_path=("Injuries")

# injury_files= glob.glob(os.path.join(injury_path, "*.csv"))

# injuries_df= glob.glob(injury_path+'\*.csv')
# injuries_df=pd.read_csv("C:/Users/E102068/Desktop/SiteRisk Dashboard/Injuries/injuries_report_Q1'24.csv")

In [ ]:
injury_files

In [ ]:
injury_df_main= pd.DataFrame()

# Appending all the csv files into one DataFrame
for file in injury_files:
    df_temp_injury = pd.read_csv(file)


    # Strip leading and trailing spaces from column names
    df_temp_injury.columns = df_temp_injury.columns.str.strip()


    # Add Quarter information
    df_temp_injury['Quarter'] = file.split('_')[-1].split('.')[0]


    # Concatenate to main DataFrame
    injury_df_main = pd.concat([injury_df_main, df_temp_injury], ignore_index=True)


In [ ]:
# v3: Updated Snowflake injury query - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
snowflake_pull_df = db_snowflake.execute(
    """
SELECT CLAIM_NUMBER as CASE_NUMBER,
       OCCURRENCE_NUMBER,
       STATUS,
       OCCURRED_DAY,
       C_FIRSTDAYOFMONTHDATE,
       LOCATION_NAME as WORK_LOCATION,
       EMPLOYEE_ID_C as EMPLOYEE_ID,
       TITLE as JOB_PROFILE,
       DEPARTMENT,
       JOB_FAMILY_GROUP,
       JOB_FAMILY,
       BODY_PART as BODYPART2,
       DATE_TRUNC('QUARTER', OCCURRED_DAY) AS DATE_QUARTER,
       CASE
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'
        END as Quarter
FROM CARLEGAL.WC_INJURY.WC_CLAIMS_INPUT
WHERE OCCURRED_DAY >= '2025-04-01' 
AND OCCURRED_DAY<'2026-01-01'

UNION ALL

-- v3: 2026+ monthly data with dynamic Month/Quarter
SELECT CLAIM_NUMBER as CASE_NUMBER,
       OCCURRENCE_NUMBER,
       STATUS,
       OCCURRED_DAY,
       C_FIRSTDAYOFMONTHDATE,
       LOCATION_NAME as WORK_LOCATION,
       EMPLOYEE_ID_C as EMPLOYEE_ID,
       TITLE as JOB_PROFILE,
       DEPARTMENT,
       JOB_FAMILY_GROUP,
       JOB_FAMILY,
       BODY_PART as BODYPART2,
       DATE_TRUNC('QUARTER', OCCURRED_DAY) AS DATE_QUARTER,
       CONCAT('Q', DATE_PART('QUARTER', OCCURRED_DAY), '''', RIGHT(DATE_PART('YEAR', OCCURRED_DAY), 2)) AS Quarter,
       TO_CHAR(OCCURRED_DAY, 'YYYY-MM') AS Month
FROM CARLEGAL.WC_INJURY.WC_CLAIMS_INPUT
WHERE OCCURRED_DAY >= '2026-01-01';

""")

In [ ]:
snowflake_pull_df= snowflake_pull_df.rename(columns={
    'QUARTER': 'Quarter',
    'JOB_FAMILY_GROUP': 'JobFamilyGroup',
    'JOB_PROFILE': 'JobProfile', 
    'JOB_FAMILY': 'JobFamily'})

In [ ]:
injury_df_main = pd.concat([injury_df_main, snowflake_pull_df], ignore_index=True)

In [ ]:
injury_df_main.groupby('Quarter').size()

In [ ]:
injury_df_main['JobFamilyGroup'].unique()

In [ ]:
# Transform the 'JobFamilyGroup' column
injury_df_main['JobFamilyGroup'] = injury_df_main['JobFamilyGroup'].apply(
    lambda x: x.upper().replace(' ', '_') if isinstance(x, str) else x
)

In [ ]:
# Pull JobProfile Detail from workday - TABLE EMPLOYEE HISTORY
JobFamily_df = db_snowflake.execute("""
                             
SELECT Distinct JOB_PROFILE,JOB_FAMILY_GROUP 
FROM PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE COMPANY NOT like '%ADESA%'
AND JOB_FAMILY_GROUP NOT like '%ADESA%';

""")

In [ ]:
JobFamily_df=JobFamily_df.rename(columns={'JOB_PROFILE':'JobProfile','JOB_FAMILY_GROUP':'JobFamilyGroup'})

JobFamily_df['JobFamilyGroup'] = np.where(JobFamily_df['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', JobFamily_df['JobFamilyGroup'])

JobFamily_df = JobFamily_df.drop_duplicates()

In [ ]:
# Merging with Workday Jobfamily to get more accurate information
injury_df_main= pd.merge(injury_df_main,JobFamily_df, how='left', on='JobProfile')

# injury_df_check= pd.merge(injury_df,JobFamily_df, how='left', left_on='JobProfile',right_on='JobProfileName')

In [ ]:
distinct_empl= db_snowflake.execute("""

WITH RankedEmployees AS (
    SELECT 
        DATE_DAY, 
        EMPLOYEE_ID, 
        LOCATION,
        ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY DATE_DAY DESC) as rn
    FROM 
        PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
    WHERE 
        LOCATION IS NOT NULL
)

SELECT  
    EMPLOYEE_ID, 
    LOCATION                   
FROM 
    RankedEmployees
WHERE 
    rn = 1
ORDER BY 
    EMPLOYEE_ID;

""")

In [ ]:
distinct_empl['EMPLOYEE_ID'] = distinct_empl['EMPLOYEE_ID'].astype(int)
injury_df_main['EMPLOYEE_ID'] = injury_df_main['EMPLOYEE_ID'].fillna(0).astype(int)

In [ ]:
injury_df_main=pd.merge(injury_df_main,distinct_empl,on='EMPLOYEE_ID',how='left')

In [ ]:
injury_df_main['WORK_LOCATION']= np.where(injury_df_main['LOCATION'].isna(),injury_df_main['WORK_LOCATION'],injury_df_main['LOCATION'])

In [ ]:
print(necessary_JobFamilyGroups)
print(injury_df_main['JobFamilyGroup_x'].unique())

In [ ]:
# Renaming the column
injury_df_main=injury_df_main.rename(columns={'JobFamilyGroup_y':'JobFamilyGroup'})


In [ ]:

# Wherever their are NA then pointing it to Company
injury_df_main['JobFamilyGroup'] = np.where(injury_df_main['JobFamilyGroup'].isna(),injury_df_main['JobFamilyGroup_x'],injury_df_main['JobFamilyGroup'])

In [ ]:
injury_df_main['JobFamilyGroup'].unique()

In [ ]:
print(necessary_JobFamilyGroups)


In [ ]:
# Define the conditions for JobRole
conditions = [
    injury_df_main['JobFamilyGroup'].str.contains('ADESA|adesa', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('RECONDITIONING|Inventory', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('WHOLESALE_OPERATIONS', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('POST_PRODUCTION', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('REGULATORY_OPERATIONS|Registration', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('LOGISTICS|TRANSPORTATION', regex=True, na=False),
    injury_df_main['JobFamilyGroup'].str.contains('MARKET_OPERATIONS', regex=True, na=False),
]


# Define the corresponding JobRole values
# values = ['ADESA','Reconditioning','Wholesale Operations','Post Production','Regulatory Operations','Logistics','Market Operations']
values = ['ADESA','Reconditioning','Wholesale Operations','Post Production','Registration','Logistics','Market Operations']


# Use np.select to create the JobRole column based on the conditions and values
injury_df_main['JobFamilyGroup'] = np.select(conditions, values, injury_df_main['JobFamilyGroup'])

In [ ]:

injury_df_main['JobFamilyGroup'] = np.where(injury_df_main['JobFamilyGroup'].isna(),injury_df_main['COMPANY'],injury_df_main['JobFamilyGroup'])


# Renaming Col
injury_df_snowflake= injury_df_main.rename(columns={'WORK_LOCATION':'Location','BODYPART2':'Injuries'})

### Snowflake - Injury

In [ ]:
injury_df_snowflake = injury_df_snowflake[['Injuries','Location','JobFamilyGroup','JobProfile','Quarter','COMPANY','STATUS','CASE_NUMBER']]

In [ ]:
injury_df_snowflake.head()

In [ ]:
injury_df_snowflake.groupby('Quarter').size()

In [ ]:
# # Uncomment when you want to run the script again
# Write into Snowflake
db_snowflake_write.insert('injuries',injury_df_snowflake,truncate=True)

In [ ]:
# # Drop the existing table

# db_snowflake_write.drop_table('Injuries')

In [ ]:
# # Table Creation

# db_snowflake_write.create_table('Injuries',
#                   {
#                      'Injuries': ['VARCHAR(100)'], 
#                      'Location': ['VARCHAR(100)'], 
#                      'JobFamilyGroup': ['VARCHAR(100)'], 
#                      'JobProfile': ['VARCHAR(100)'],
#                      'Quarter': ['VARCHAR(1000)'],
#                      'Company': ['VARCHAR(100)'], 
#                      'STATUS': ['VARCHAR(100)'],
#                      'CASE_NUMBER': ['VARCHAR(100)']
#                   })

### Injury df

In [ ]:
# Filtering Only for necessary_JobFamilyGroups
injury_df= injury_df_snowflake[injury_df_snowflake['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [ ]:
injury_df['JobFamilyGroup'].unique()

In [ ]:
# v3: Ensure Month column exists for injury data
if 'Month' not in injury_df.columns:
    injury_df['Month'] = None

# v3: Derive Month from OCCURRED_DAY for 2026+ Snowflake data
if 'OCCURRED_DAY' in injury_df.columns:
    injury_df['Month'] = derive_month_from_date(injury_df['OCCURRED_DAY'])

# Counting the number of injuries at the Location-Org Combination
# v3: Include Month in groupby for monthly granularity
injury_df = injury_df.groupby(['Location','JobFamilyGroup','Quarter','Month']).agg('count').reset_index()

# Final Injury df
injury_df = injury_df[['Location', 'JobFamilyGroup','Quarter','Month','Injuries']]

## Main Dataframe

In [ ]:
# v3: Merging for Specific CostCentre - include Month in merge keys
main_df= pd.merge(location_df,injury_df, how='outer', on=['Location','JobFamilyGroup','Quarter','Month'])

# Performance PIPs/CARs

In [ ]:
# Performance for Last Quater
performance_sql_df = pd.read_sql_query(f"""

WITH cte AS (
Select * FROM PeopleOps.workday.FactEmployeeReviews
WHERE 
--DateInitiated >= DATEADD(QUARTER, DATEDIFF(QUARTER, 0, '') - 1, 0)
--AND DateInitiated < DATEADD(QUARTER, DATEDIFF(QUARTER, 0, ''), 0) 
--AND Status in ('In Progress','Successfully Completed','Rescinded') 
ReviewCategory in ('Corrective Action','Performance Improvement Plan')
AND DateInitiated>='2024-01-01'
AND DateInitiated < DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()), 0))

SELECT c.EmployeeId,c.ReviewCategory,c.ReviewReason,c.ResultingActionRating,c.DateInitiated,c.Status,
eh.StartDateTime,eh.EndDateTime,eh.JobFamilyGroup,eh.Company,eh.Location,
CONCAT('Q', DATEPART(quarter,c.DateInitiated ), '''', RIGHT(DATEPART(year, c.DateInitiated), 2)) as Quarter   
FROM cte c left join PeopleOps.workday.tblEmployeeHistory eh ON c.EmployeeId=eh.EmployeeId
WHERE c.DateInitiated between eh.StartDateTime and eh.EndDateTime
ORDER BY c.DateInitiated

""",odbcconn)

In [ ]:
## Snowflake Pull for Performance Data

performance_snowflake_base_df = db_snowflake.execute("""
                                                     
with cte as (
SELECT *
FROM PEOPLEOPS.WORKDAY.VW_EMPLOYEE_REVIEWS
WHERE REVIEW_CATEGORY IN ('Corrective Action' ,'Performance Improvement Plan')
AND DATE_INITIATED::DATE>='2024-01-01'
AND DATE_INITIATED::DATE <= LAST_DAY(DATEADD(quarter, -1, DATE_TRUNC(quarter, CURRENT_DATE)),'quarter')
)

SELECT c.EMPLOYEE_ID
,c.REVIEW_CATEGORY
,c.REVIEW_REASON
,c.RESULTING_ACTION_RATING
,c.DATE_INITIATED
,c.STATUS
,eh.START_TIME
,eh.END_TIME
,eh.JOB_FAMILY_GROUP
,eh.COMPANY
,eh.LOCATION_NAME AS LOCATION
,CONCAT('Q',DATE_PART('QUARTER',c.DATE_INITIATED),'''',RIGHT(DATE_PART('YEAR',C.DATE_INITIATED),2)) AS QUARTER
FROM CTE c
LEFT JOIN PEOPLEOPS.WORKDAY.VW_DIM_EMPLOYEE_HISTORY eh 
ON c.EMPLOYEE_ID = eh.EMPLOYEE_ID
WHERE c.DATE_INITIATED BETWEEN eh.START_TIME AND eh.END_TIME
ORDER BY C.DATE_INITIATED;
                                                     
""")

In [ ]:
date_today

### In progress performance 

In [ ]:
# Inprogress List with ToDO step COmpleted 

performance_inprogress_sql_df= pd.read_sql_query(f"""

SELECT EmployeeId, Manager, ReviewTemplate, ProcessStatus as Status, ProcessInitiationDate as DateInitiated,
Step,StepStatus,StepCompletedOn,StepCompletedBy 
FROM PeopleOps.workday.FactEmployeeReviewProcess
WHERE 
--ProcessInitiationDate >= DATEADD(QUARTER, DATEDIFF(QUARTER, 0, '{date_today}') - 1, 0)
ProcessInitiationDate < DATEADD(QUARTER, DATEDIFF(QUARTER, 0, '{date_today}'), 0)
AND ProcessInitiationDate>='2024-01-01'
AND Step like '%To Do%' 
AND StepStatus = 'Step Completed'
AND ProcessStatus ='In Progress'
ORDER BY EmployeeId

""",odbcconn)

In [ ]:
performance_inprogress_snowflake_base_df = db_snowflake.execute(f"""
SELECT EMPLOYEE_ID
,MANAGER
,REVIEW_TEMPLATE
,PROCESS_STATUS as STATUS
,PROCESS_INITIATION_DATE as DATE_INITIATED
,STEP
,STEP_STATUS
,STEP_COMPLETED_ON
,STEP_COMPLETED_BY
FROM PEOPLEOPS.WORKDAY.VW_EMPLOYEE_REVIEW_PROCESS
WHERE PROCESS_INITIATION_DATE  >='2024-01-01'
AND PROCESS_INITIATION_DATE <= LAST_DAY(DATEADD(quarter, -1, DATE_TRUNC(quarter, CURRENT_DATE)),'quarter')
AND STEP LIKE '%To Do%'
AND STEP_STATUS ='Step Completed'
AND PROCESS_STATUS = 'In Progress'
ORDER BY PROCESS_INITIATION_DATE desc;
""")

In [ ]:
# performance_sql_df.value_counts('Status')

In [ ]:
performance_snowflake_base_df[performance_snowflake_base_df['QUARTER']=="Q4'25"].value_counts('STATUS')

In [ ]:
# Creating Keys to check if the value exist in inprogess df (can be done using merge)

# performance_sql_df['key']=performance_sql_df['EmployeeId'].astype(str)+performance_sql_df['Status'].astype(str)\
#     +performance_sql_df['DateInitiated'].astype(str)
# performance_inprogress_sql_df['key']=performance_inprogress_sql_df['EmployeeId'].astype(str)+performance_inprogress_sql_df['Status'].astype(str)\
#     +performance_inprogress_sql_df['DateInitiated'].astype(str)

# # Finding rows that exist in inprogress and updating their status
# performance_sql_df.loc[performance_sql_df['key'].isin(performance_inprogress_sql_df['key']), 'Status'] = 'Successfully Completed'


performance_snowflake_base_df['key']=performance_snowflake_base_df['EMPLOYEE_ID'].astype(str)+performance_snowflake_base_df['STATUS'].astype(str)\
    +performance_snowflake_base_df['DATE_INITIATED'].astype(str)
performance_inprogress_snowflake_base_df['key']=performance_inprogress_snowflake_base_df['EMPLOYEE_ID'].astype(str)+performance_inprogress_snowflake_base_df['STATUS'].astype(str)\
    +performance_inprogress_snowflake_base_df['DATE_INITIATED'].astype(str)

# Finding rows that exist in inprogress and updating their status
performance_snowflake_base_df.loc[performance_snowflake_base_df['key'].isin(performance_inprogress_snowflake_base_df['key']), 'STATUS'] = 'Successfully Completed'

In [ ]:
# drop the key column
performance_snowflake_base_df = performance_snowflake_base_df.drop(columns=['key'])
performance_inprogress_snowflake_base_df = performance_inprogress_snowflake_base_df.drop(columns=['key'])

In [ ]:
performance_snowflake_base_df.value_counts('STATUS')

### ADESA PIPs/CARs

In [ ]:
# Converting Job Families in company ADESA as ADESA 
# performance_sql_df['JobFamilyGroup'] = np.where(performance_sql_df['Company'].str.contains('ADESA'),"ADESA",performance_sql_df['JobFamilyGroup'])

performance_snowflake_base_df['JOB_FAMILY_GROUP'] = np.where(performance_snowflake_base_df['COMPANY'].str.contains('ADESA'),"ADESA",performance_snowflake_base_df['JOB_FAMILY_GROUP'])


In [ ]:
# Neccessary Job Family Group
# Filtering for siterisk costcentres
performance_snowflake_base_df=performance_snowflake_base_df[performance_snowflake_base_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups)]

In [ ]:
performance_snowflake_base_df.value_counts('STATUS')

### Snowflake - Performance 

In [ ]:
performance_snowflake_df = performance_snowflake_base_df.copy()

In [ ]:
performance_snowflake_df.head()

In [ ]:
performance_snowflake_df.groupby('QUARTER').size()  

In [ ]:
performance_snowflake_df.describe()

In [ ]:
performance_snowflake_df['JOB_FAMILY_GROUP'] = np.where(performance_snowflake_df['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', performance_snowflake_df['JOB_FAMILY_GROUP'])

In [ ]:
# Uncomment when you need to run it again
# Write into Snowflake
# db_snowflake_write.drop_table('Performance')
# db_snowflake_write.dump('Performance',performance_snowflake_df)
db_snowflake_write.insert('Performance',performance_snowflake_df,truncate=True)

In [ ]:
# # Drop the existing table

# db_snowflake_write.drop_table('Performance')

In [ ]:
# # Performance Table Creation

# db_snowflake_write.create_table('Performance',
#                   {
#                      'EmployeeId': ['INT','NOT NULL'],
#                      'ReviewCategory': ['VARCHAR(1000)'], 
#                      'ReviewReason': ['VARCHAR(1000)'], 
#                      'ResultingActionRating': ['VARCHAR(1000)'],
#                      'Status':['VARCHAR(1000)'],
#                      'DateInitiated': ['TIMESTAMP_NTZ(9)'], 
#                      'StartDateTime': ['TIMESTAMP_NTZ(9)'],
#                      'EndDateTime' : ['TIMESTAMP_NTZ(9)'],
#                      'JobFamilyGroup': ['VARCHAR(1000)'],
#                      'Company': ['VARCHAR(1000)'],
#                      'Location': ['VARCHAR(1000)'],
#                      'Quarter': ['VARCHAR(1000)']
#                   })

### Performance Df

In [ ]:
performance_df= db_snowflake.execute("""

SELECT * FROM PEOPLEOPS_SANDBOX.SITERISK.PERFORMANCE

""")

In [ ]:
# Filtering only for Successfully completed
performance_df=performance_df[performance_df['STATUS']=='Successfully Completed']

In [ ]:
performance_df.value_counts('STATUS')

In [ ]:
performance_df['QUARTER'].unique()

In [ ]:
# Selecting neccesary columns
# performance_df= performance_df.iloc[:,[0,1,-4,-2,-1]]
performance_df= performance_df[['EMPLOYEE_ID','LOCATION','JOB_FAMILY_GROUP','QUARTER','REVIEW_CATEGORY']]

In [ ]:
# Count the number of PIPs/Cars and Unique Employees
performance_df= performance_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER','REVIEW_CATEGORY']).\
                                agg(Total_Count=('EMPLOYEE_ID','count'),EmployeesPIPs_CARs=('EMPLOYEE_ID','nunique')).reset_index()

In [ ]:
performance_df['Total_Count'].sum()

In [ ]:
# Pivot to divide the PIPs/CARs
performance_df= performance_df.pivot(index=['LOCATION','JOB_FAMILY_GROUP','QUARTER','EmployeesPIPs_CARs'],\
                                     columns=['REVIEW_CATEGORY'],values='Total_Count').reset_index()

In [ ]:
# Total sum of all
performance_df= performance_df.groupby(['LOCATION','JOB_FAMILY_GROUP','QUARTER']).agg('sum').reset_index()

## Overall PIPs/CARs

In [ ]:
# Filling all NA with 0 for addition
performance_df=performance_df.fillna(0)

# Sum of ALL PIPs/CARs
performance_df['PIPs & CARs']=performance_df['Performance Improvement Plan']\
                                    +performance_df['Corrective Action']

# Rename
performance_df=performance_df.rename(columns={'Performance Improvement Plan':'PIPs',\
                                              'Corrective Action':'CARs'})

In [ ]:
performance_df['PIPs & CARs'].sum()

In [ ]:
#Rename Colmns
performance_df=performance_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter'})

In [ ]:
performance_df['JobFamilyGroup'].unique()

## Merge-Main Df

In [ ]:
# v3: Add Month to performance_df before merge (derive from DATE_INITIATED for 2026+)
if 'Month' not in performance_df.columns:
    performance_df['Month'] = None
main_df=pd.merge(main_df,performance_df,on=['Location','Quarter','JobFamilyGroup','Month'],how='outer')

In [ ]:
main_df.head()

# Turnover/Attrition

Update the Quater based on when you are running****

In [ ]:
## v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter

turnover_sql_terms= db_snowflake_write.execute(f"""

SELECT 
    DATE_DAY as TerminationDate,
    EMPLOYEE_ID,
    EMPLOYEE_NAME as NAME,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP, 
    LOCATION, 
    CASE
        WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
        WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
        WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
        WHEN DATE_QUARTER = '2024-10-01' THEN 'Q4''24'
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'                                              
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'                                    
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                            
    END AS Quarter,
    NULL AS Month
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'TERMINATED'
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2024-01-01'
    AND DATE_DAY<'2026-01-01'

UNION ALL

-- v3: 2026+ monthly data with dynamic Month/Quarter
SELECT 
    DATE_DAY as TerminationDate,
    EMPLOYEE_ID,
    EMPLOYEE_NAME as NAME,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP, 
    LOCATION, 
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'TERMINATED'
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2026-01-01'

ORDER BY 
    TerminationDate

""")

In [ ]:
turnover_sql_terms['JOB_FAMILY_GROUP'].unique()

In [ ]:
turnover_sql_terms['JOB_FAMILY_GROUP'] = np.where(turnover_sql_terms['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', turnover_sql_terms['JOB_FAMILY_GROUP'])

In [ ]:
# Converting to Date
turnover_sql_terms['TERMINATIONDATE']=pd.to_datetime(turnover_sql_terms['TERMINATIONDATE'])

# Neccesary JobFamilyGroups
turnover_sql_terms=turnover_sql_terms[turnover_sql_terms['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & turnover_sql_terms['QUARTER'].notna()]

# # Adding Quarter based on Date column
# turnover_sql_terms['Quarter'] = turnover_sql_terms['TerminationDate'].dt.to_period("Q")
# turnover_sql_terms['Quarter'] = turnover_sql_terms['Quarter'].apply(lambda q: f"{q.strftime('Q%q')}'{str(q.year)[2:]}")

### Snowflake - Terminated Emp

In [ ]:
#Create Table
# db_snowflake_write.create_table('TermEmployee',
#                   {
#                      'EmployeeId': ['INT','NOT NULL'],
#                      'Name': ['VARCHAR(1000)'], 
#                      'JobFamilyGroup': ['VARCHAR(1000)'], 
#                      'Location': ['VARCHAR(1000)'],
#                      'TerminationDate': ['TIMESTAMP_NTZ(9)'], 
#                      'Quarter': ['VARCHAR(1000)']
#                   })

In [ ]:
# db_snowflake_write.drop_table('TermEmployee')

In [ ]:
print(turnover_sql_terms.shape)
turnover_sql_terms.head()

In [ ]:
turnover_sql_terms.groupby('QUARTER').size()

**RUN THIS WRITE CAREFULLY

In [ ]:
#Insert Data

db_snowflake_write.insert('TermEmployee',turnover_sql_terms,truncate=True)

### Location/JobFamilyGroup Calculation

#### Active Emp Calculation

In [ ]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
active_df=db_snowflake.execute("""

-- Pre-2026: quarterly with CASE statements
SELECT
    DATE_DAY as DATE,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CASE
        WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
        WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
        WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
        WHEN DATE_QUARTER = '2024-10-01' THEN 'Q4''24'
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                               
    END AS Quarter,
    NULL AS Month,
    LOCATION,
    COUNT(*) as Active_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE in('ACTIVE','ON LEAVE')
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2024-01-01'
    AND DATE_DAY<'2026-01-01'
GROUP BY DATE_DAY,LOCATION,DATE_QUARTER,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

UNION ALL

-- v3: 2026+ monthly data with dynamic Month/Quarter
SELECT
    DATE_DAY as DATE,
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month,
    LOCATION,
    COUNT(*) as Active_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE in('ACTIVE','ON LEAVE')
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY>='2026-01-01'
GROUP BY DATE_DAY,LOCATION,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

ORDER BY DATE,JOB_FAMILY_GROUP,LOCATION,Active_employees;

""")

In [ ]:
# active_df['JOB_FAMILY_GROUP'] = np.where(active_df['JOB_FAMILY_GROUP'].str.contains('Regulatory'), 'Registration', active_df['JOB_FAMILY_GROUP'])

In [ ]:
active_df=active_df[active_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & active_df['QUARTER'].notna()]

In [ ]:
# # Count the number of active employees on each day
# active_df= active_emps.groupby(['DATE','JOB_FAMILY_GROUP','LOCATION','QUARTER'])\
#     .agg(employees=('EMPLOYEE_ID','count')).reset_index()

# # Adding WeekNumber for based on Date column
# active_df['WeekNumber']=active_df['Date'].dt.isocalendar().week # Add this for weekly turnover

# # # Adding Quarter based on Date column
# # active_df['Quarter'] = active_df['Date'].dt.to_period("Q")
# # active_df['Quarter'] = active_df['Quarter'].apply(lambda q: f"{q.strftime('Q%q')}'{str(q.year)[2:]}")

In [ ]:
# Quarter calculation
# Calculating the average headcount for the Quater
# (Add Week Number in groupby for weekly turnover)

active_qt_df=active_df.groupby(['QUARTER','JOB_FAMILY_GROUP','LOCATION'])\
    .agg(Avg_emp=('ACTIVE_EMPLOYEES','mean')).reset_index()

# Rounding the weekly
active_qt_df['Avg_emp']=round(active_qt_df['Avg_emp'],2)

#### Term Calculation

In [ ]:
turnover_terms= db_snowflake.execute("""

SELECT * FROM PEOPLEOPS_SANDBOX.SITERISK.TERMEMPLOYEE

""")

In [ ]:
# Count number of terms each day
term_df= turnover_terms.groupby(['TERMINATIONDATE','JOB_FAMILY_GROUP','LOCATION','QUARTER'])\
    .agg(Terminations=('EMPLOYEE_ID','count')).reset_index()

# # Renaming Termination Date
# term_df=term_df.rename(columns={'TERMINATIONDATE':'Date'})

# # Adding Weeknumber based on Termination Date
# term_df['WeekNumber']=term_df['Date'].dt.isocalendar().week # Add this for weekly turnover

# # Adding Quarter based on Date column
# term_df['Quarter'] = term_df['Date'].dt.to_period("Q")
# term_df['Quarter'] = term_df['Quarter'].apply(lambda q: f"{q.strftime('Q%q')}'{str(q.year)[2:]}")

In [ ]:
# Quarter calculation
# Calculating the terms for the quarter

term_qt_df=term_df.groupby(['QUARTER','JOB_FAMILY_GROUP','LOCATION'])\
    .agg(Terminations=('Terminations','sum')).reset_index()

#### Turnover

In [ ]:
# # Combining Active and Terms 
# turnover_df=pd.merge(active_df,term_df,on=['Date','Quarter','Location','JobFamilyGroup'],how='left')

# # Fill NA
# turnover_df=turnover_df.fillna(0)

#### Quaterly Turnover

In [ ]:
# Combining Active and Terms 
turnover_qt_df=pd.merge(active_qt_df,term_qt_df,on=['QUARTER','LOCATION','JOB_FAMILY_GROUP'],how='left')

# Fill NA
turnover_qt_df=turnover_qt_df.fillna(0)

In [ ]:
turnover_qt_df= turnover_qt_df.groupby(['QUARTER','JOB_FAMILY_GROUP','LOCATION']).\
                            agg(Avg_emp=('Avg_emp','mean'),\
                                Terminations=('Terminations','sum')).reset_index()

# Calculating the turnover rate 
turnover_qt_df['Quaterly_Turnover']=round((turnover_qt_df['Terminations']/turnover_qt_df['Avg_emp'])*100,2)

In [ ]:
turnover_qt_df=turnover_qt_df.rename(columns={'QUARTER':'Quarter','JOB_FAMILY_GROUP':'JobFamilyGroup','LOCATION':'Location'})

## Change

In [ ]:
import re
# Function to parse quarter and year
def parse_quarter(quarter_str):
    match = re.match(r"Q([1-4])'(\d{2})", quarter_str)
    if match:
        return int(match.group(2)) * 4 + int(match.group(1))
    else:
        return None

In [ ]:
# Add a new column for sorting
turnover_qt_df['sort_key'] = turnover_qt_df['Quarter'].apply(parse_quarter)


# Ensure the DataFrame is sorted by location, jobfamily, and sort_key
turnover_qt_df = turnover_qt_df.sort_values(by=['Location', 'JobFamilyGroup', 'sort_key'])


# Calculate the change in turnover for consecutive quarters
turnover_qt_df['Turnover_change'] = turnover_qt_df.groupby(['Location', 'JobFamilyGroup'])['Quaterly_Turnover'].diff()


# Drop the temporary sort_key column
turnover_qt_df = turnover_qt_df.drop(columns=['sort_key'])

In [ ]:
turnover_qt_df['Turnover_change']=turnover_qt_df['Turnover_change'].fillna(0).round(2)

In [ ]:
turnover_qt_df['JobFamilyGroup'].unique()

## Merge - Main DF

In [ ]:
# v3: Add Month to turnover_qt_df before merge
if 'Month' not in turnover_qt_df.columns:
    turnover_qt_df['Month'] = None
main_df=pd.merge(main_df,turnover_qt_df,on=['Quarter','JobFamilyGroup','Location','Month'],how='outer')


# Earlier left join but now want to capture all the active employees even if they dont have turnover data for the quarter so changing to outer join
# main_df=pd.merge(main_df,turnover_qt_df,on=['Quarter','JobFamilyGroup','Location'],how='left')

# Removing avg_emp col
# main_df.drop('Avg_emp', axis=1, inplace=True)

In [ ]:
# main_df=main_df.drop(columns=['Terminations','Quaterly_Turnover','Turnover_change'])

# Navex - Siterisk 2.0

In [ ]:
## SITE RISK 2.0
navex_files

In [ ]:
valid_locations = main_df[main_df['Score'] > 0][['Location', 'JobFamilyGroup']].drop_duplicates().to_dict('records')
# valid_locations = main_df[['Location', 'JobFamilyGroup']].drop_duplicates().to_dict('records')


print(valid_locations[:5]) # Print the first 5 to check the structure

In [ ]:
# v3: Single Navex file - derive Quarter/Month from MONTH_OPENED column
navex_df_main = pd.read_csv(navex_files[0])

# Strip leading and trailing spaces from column names
navex_df_main.columns = navex_df_main.columns.str.strip()

# Parse MONTH_OPENED (format: MM/DD/YYYY) to derive Quarter and Month
navex_df_main['MONTH_OPENED'] = pd.to_datetime(navex_df_main['MONTH_OPENED'], format='%m/%d/%Y')
navex_df_main['Quarter'] = derive_quarter_from_date(navex_df_main['MONTH_OPENED'])
navex_df_main['Month'] = derive_month_from_date(navex_df_main['MONTH_OPENED'])

In [ ]:
navex_df_main= navex_df_main.drop(columns=['LATITUTDE','LONGITUDE'])
navex_df_main=navex_df_main.drop_duplicates()
navex_df_main=navex_df_main.rename(columns={'BRANCH_NUMBER':'JobFamilyGroup'})

In [ ]:
navex_df_main['JobFamilyGroup'].unique()

In [ ]:
navex_df_main['JobFamilyGroup'] = np.where(navex_df_main['JobFamilyGroup'].str.contains('Regulatory'), 'Registration', navex_df_main['JobFamilyGroup'])

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process


def map_location(row, valid_locations, min_score=80):
    """
    Maps a given location name to the best matching valid location
    within the same job family group, prioritizing specific conditions.

    Args:
        row (pd.Series): A row from the DataFrame containing 'LOCATION_NAME' and 'JOBFAMILYGROUP'.
        valid_locations (list): A list of dictionaries with 'Location' and 'JobFamilyGroup'.
        min_score (int): Minimum fuzzy match score (0-100) to consider a match.

    Returns:
        str: The best matching valid location, or None if no good match is found.
    """
    location_name = row['LOCATION_NAME']
    job_family = row['JobFamilyGroup']

    if pd.isna(location_name):
        return None

    location_name_upper = location_name.upper()

    if location_name == 'Remote':
        return 'Remote'
    elif location_name == 'Other':
        return 'Other'
    elif 'VM' in location_name_upper or 'ADESA' in location_name_upper or 'VLC' in location_name_upper:
        return location_name
    else:
        # Filter valid locations by job family group
        relevant_locations = [
            loc['Location'] for loc in valid_locations if loc['JobFamilyGroup'] == job_family
        ]
        if not relevant_locations:
            return None  # No valid locations for this job family

        best_match, score = process.extractOne(location_name, relevant_locations, scorer=fuzz.token_set_ratio)
        if score >= min_score:
            return best_match
        return None

# Assuming your DataFrame 'df' has columns 'LOCATION_NAME' and 'JOBFAMILYGROUP'
navex_df_main['Mapped_Location'] = navex_df_main.apply(map_location, axis=1, valid_locations=valid_locations)


In [ ]:
navex_df_main[['LOCATION_NAME', 'JobFamilyGroup', 'Mapped_Location']]

In [ ]:
navex_df_main['Mapped_Location']= np.where(navex_df_main['Mapped_Location'].isna(),navex_df_main['LOCATION_NAME'],navex_df_main['Mapped_Location'])

In [ ]:
navex_df_main.head()

In [ ]:
navex_df_main.value_counts('Quarter')

In [ ]:
navex_df_main.columns

In [ ]:
navex_df_main=navex_df_main.rename(columns={'LOCATION_NAME':'Old_Location','Mapped_Location':'Location'})

navex_df_main['JobFamilyGroup'].unique()

In [ ]:
# navex_df_main= navex_df_main.drop(columns=['QUARTER'])

## Snowflake

In [ ]:
# Snowflake Dump
navex_df_snowflake= navex_df_main.copy()
db_snowflake_write.drop_table('Navex')
db_snowflake_write.dump('Navex',navex_df_snowflake)

## Navex Df

In [ ]:
navex_df= navex_df_main.copy()
navex_df= navex_df[navex_df['JobFamilyGroup'].isin(necessary_JobFamilyGroups)]

In [ ]:
# v3: Include Month in groupby for monthly granularity
if 'Month' not in navex_df.columns:
    navex_df['Month'] = None
navex_df= navex_df.groupby(['Location','JobFamilyGroup','Quarter','Month'])\
    .agg(Navex_Incidents=('CASE_NUMBER','nunique')).reset_index()
    # .agg(Navex_Incidents=('CASE_STATUS','count')).reset_index()

In [ ]:
navex_df.value_counts('Quarter')

In [ ]:
navex_df['Navex_Incidents'].sum()

In [ ]:
navex_df.head()

In [ ]:
navex_df['JobFamilyGroup'].unique()

## Main df

In [ ]:
# v3: Include Month in Navex merge
main_df=pd.merge(main_df,navex_df,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')
# main_df['Navex_Incidents']=main_df['Navex_Incidents'].fillna(0)

# Total Employees- Location/JobProfile

*** ADD QUARTER DATA IN 'CASE' STATEMENT

In [ ]:
# v3: Updated - pre-2026 uses day 89 of quarter, 2026+ uses last day of month
totalemp_Loc_Jbp_df = db_snowflake.execute("""

-- Pre-2026: quarterly snapshot (day 89 of quarter)
SELECT
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CASE
        WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
        WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
        WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
        WHEN DATE_QUARTER = '2024-10-01' THEN 'Q4''24'
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                   
    END AS Quarter,
    NULL AS Month,
    LOCATION,
    COUNT(*) as Total_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'ACTIVE'
    AND EMPLOYEE_TYPE='Regular'
    AND DATEDIFF(DAY,DATE_QUARTER,DATE_DAY)=89
    AND DATE_DAY>='2024-01-01'
    AND DATE_DAY<'2026-01-01'
GROUP BY DATE_DAY,LOCATION,DATE_QUARTER,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

UNION ALL

-- v3: 2026+ monthly snapshot (last day of each month)
SELECT
    CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP
    END as JOB_FAMILY_GROUP,
    CONCAT('Q', DATE_PART('QUARTER', DATE_DAY), '''', RIGHT(DATE_PART('YEAR', DATE_DAY), 2)) AS Quarter,
    TO_CHAR(DATE_DAY, 'YYYY-MM') AS Month,
    LOCATION,
    COUNT(*) as Total_employees
FROM 
    PEOPLEOPS.ANALYTICS.VW_WORKDAY_EMPLOYEE_LINEAGE
WHERE 
    DATA_TYPE = 'ACTIVE'
    AND EMPLOYEE_TYPE='Regular'
    AND DATE_DAY = LAST_DAY(DATE_DAY, 'month')
    AND DATE_DAY>='2026-01-01'
GROUP BY DATE_DAY,LOCATION,
CASE
    WHEN JOB_FAMILY_GROUP like '%ADESA%' THEN 'ADESA'
    WHEN JOB_FAMILY_GROUP like '%Regulatory%' THEN 'Registration'
    ELSE JOB_FAMILY_GROUP END

ORDER BY Quarter,JOB_FAMILY_GROUP,LOCATION,Total_employees;

""")

In [ ]:
totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df.drop_duplicates()

In [ ]:
totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df[totalemp_Loc_Jbp_df['JOB_FAMILY_GROUP'].isin(necessary_JobFamilyGroups) & totalemp_Loc_Jbp_df['QUARTER'].notna()]

totalemp_Loc_Jbp_df=totalemp_Loc_Jbp_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','TOTAL_EMPLOYEES':'Total_Employees'})

In [ ]:
totalemp_Loc_Jbp_df.groupby('Quarter').size()

In [ ]:
totalemp_Loc_Jbp_df['JobFamilyGroup'].unique()

In [ ]:
# total_emp_ADESA= db_snowflake.execute("""

# SELECT 'ADESA' as JOB_FAMILY_GROUP,LOCATION,
#     CASE
#         WHEN DATE_QUARTER = '2024-01-01' THEN 'Q1''24'
#         WHEN DATE_QUARTER = '2024-04-01' THEN 'Q2''24'
#         WHEN DATE_QUARTER = '2024-07-01' THEN 'Q3''24'
#         ELSE 'Unknown'
#     END AS Quarter
#     ,Count(*) as Total_Employees
# FROM PEOPLEOPS_SANDBOX.DEV.WORKDAY_EMPLOYEE_LINEAGE
# WHERE DATA_TYPE='ACTIVE'
# AND 
# AND DATEDIFF(DAY,DATE_QUARTER,DATE_DAY)=90
# AND DATE_DAY>='2024-01-01'
# AND JOB_PROFILE Like '%ADESA%'
# GROUP BY LOCATION,DATE_QUARTER
# ORDER BY Total_Employees desc,
# LOCATION;

# """)

In [ ]:
# totalemp_Loc_Jbp_df=pd.concat([totalemp_Loc_Jbp_df,total_emp_ADESA])

## Active Qt Avg Emp

In [ ]:
# active_qt_df=active_qt_df.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','Avg_emp':'Average_Headcount'})

## Merge- Main df

In [ ]:
# v3: Add Month to totalemp before merge
if 'Month' not in totalemp_Loc_Jbp_df.columns:
    totalemp_Loc_Jbp_df['Month'] = None
main_df= pd.merge(main_df,totalemp_Loc_Jbp_df,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

In [ ]:
main_df['Location']=main_df['Location'].astype(str)

In [ ]:
# main_df=main_df.drop(columns=['DATE'])

In [ ]:
# main_df=pd.merge(main_df,active_qt_df,on=['Location','JobFamilyGroup','Quarter'],how='left')

In [ ]:
main_df['PIPs & CARs'].sum()

# Leadership

In [ ]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter
leadership = db_snowflake.execute(""" 
with cte as (
SELECT DATE_WEEK,LOCATION,JOB_FAMILY_GROUP
,SUM(TOTAL_LEADERS) as TOTAL_LEADERS
,SUM(NEW_LEADERS) as NEW_LEADERS
,SUM(LEADER_HIRED) as LEADER_HIRED
,SUM(TERMINATIONS) as LEADER_TERMINATIONS
,SUM(OPENINGS) as OPENINGS
FROM PEOPLEOPS_SANDBOX.SITERISK.LEADERSHIP
GROUP BY ALL
)
,
-- Pre-2026: quarterly with CASE statements
pre_2026 as (
SELECT
DATE_TRUNC('quarter',DATE_WEEK) as DATE_QUARTER
,    CASE
        WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
        WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
        WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
        WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                          
    END AS Quarter
, NULL AS Month
,LOCATION
,JOB_FAMILY_GROUP
,COALESCE(ROUND(AVG(TOTAL_LEADERS),0),0) as AVG_TOTAL_LEADERS
,COALESCE(ROUND(AVG(NEW_LEADERS),0),0) as AVG_NEW_LEADERS
,COALESCE(ROUND(AVG(OPENINGS),0),0) as AVG_OPENINGS
, ROUND(DIV0(AVG_NEW_LEADERS,AVG_TOTAL_LEADERS)*100,1) as PERCENTAGE_NEW_LEADER
, ROUND(DIV0(AVG_TOTAL_LEADERS,(AVG_TOTAL_LEADERS + AVG_OPENINGS))*100,1) as PERCENTAGE_STAFFED
FROM cte
WHERE DATE_WEEK < '2026-01-01'
AND QUARTER IS NOT NULL
GROUP BY ALL
)
,
-- v3: 2026+ monthly data with dynamic Month/Quarter
post_2026 as (
SELECT
DATE_TRUNC('quarter',DATE_WEEK) as DATE_QUARTER
, CONCAT('Q', DATE_PART('QUARTER', DATE_WEEK), '''', RIGHT(DATE_PART('YEAR', DATE_WEEK), 2)) AS Quarter
, TO_CHAR(DATE_WEEK, 'YYYY-MM') AS Month
,LOCATION
,JOB_FAMILY_GROUP
,COALESCE(ROUND(AVG(TOTAL_LEADERS),0),0) as AVG_TOTAL_LEADERS
,COALESCE(ROUND(AVG(NEW_LEADERS),0),0) as AVG_NEW_LEADERS
,COALESCE(ROUND(AVG(OPENINGS),0),0) as AVG_OPENINGS
, ROUND(DIV0(AVG_NEW_LEADERS,AVG_TOTAL_LEADERS)*100,1) as PERCENTAGE_NEW_LEADER
, ROUND(DIV0(AVG_TOTAL_LEADERS,(AVG_TOTAL_LEADERS + AVG_OPENINGS))*100,1) as PERCENTAGE_STAFFED
FROM cte
WHERE DATE_WEEK >= '2026-01-01'
GROUP BY ALL
)
SELECT * FROM pre_2026
UNION ALL
SELECT * FROM post_2026;
                                   """)

In [ ]:
leadership.head()

In [ ]:
leadership.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter'},inplace=True)

In [ ]:
leadership.columns

In [ ]:
# v3: Add Month to leadership before merge
if 'Month' not in leadership.columns:
    leadership['Month'] = None
main_df=pd.merge(main_df,leadership[['Location','JobFamilyGroup','Quarter','Month','AVG_TOTAL_LEADERS', 'AVG_NEW_LEADERS', 'AVG_OPENINGS','PERCENTAGE_NEW_LEADER', 'PERCENTAGE_STAFFED']]\
                 ,on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

In [ ]:
# main_df.rename(columns={'AVG_LEADERS':'Avg_Leaders', 'LEADER_HIRED':'Leader_Hired', 'LEADER_TERMED':'Leader_Termed'}, inplace=True)

# Overtime

In [ ]:
# overtime_snow = db_snowflake.execute("""
#     SELECT 
#         LOCATIONNAME as LOCATION
#         ,DATE_TRUNC('quarter',DATE) AS DATE_QUARTER
#         ,CASE
#         WHEN DATE_QUARTER = '2025-01-01' THEN 'Q1''25'                                                                           
#         WHEN DATE_QUARTER = '2025-04-01' THEN 'Q2''25'
#         WHEN DATE_QUARTER = '2025-07-01' THEN 'Q3''25'
#         WHEN DATE_QUARTER = '2025-10-01' THEN 'Q4''25'                                                                          
#         END AS Quarter
#         ,'Reconditioning' AS JOB_FAMILY_GROUP
#         ,ROUND(SUM(OVERTIMEHOURS),0) as TOTAL_OVERTIME
#     FROM PEOPLEOPS_SANDBOX.DEV.IC_OVERTIME_AVG_HOURS
#     WHERE DATE>='2025-01-01'
#     GROUP BY LOCATIONNAME,DATE_TRUNC('quarter',DATE);
#                                      """)


### UPDATE TO SNOWFLAKE VERSION TABLE [INVENTORY.PUBLIC.EXPECTED_VS_PAID_HOURS]

In [ ]:
# v3: Updated - pre-2026 uses CASE, 2026+ uses dynamic Month/Quarter (SQL Server syntax)
overtimequery = """Select LocationName as LOCATION,
DATETRUNC(QUARTER,Date) as DATE_QUARTER,
CASE 
 WHEN JobFamilyGroup like '%RECON%' Then 'Reconditioning'
 WHEN JobFamilyGroup like '%POST%' Then 'Post Production'
 ELSE NULL
 END
 AS JOB_FAMILY_GROUP
,CASE
         WHEN DATETRUNC(QUARTER,Date) = '2025-01-01' THEN 'Q1''25'                                                                           
         WHEN DATETRUNC(QUARTER,Date) = '2025-04-01' THEN 'Q2''25'
         WHEN DATETRUNC(QUARTER,Date) = '2025-07-01' THEN 'Q3''25'
         WHEN DATETRUNC(QUARTER,Date) = '2025-10-01' THEN 'Q4''25'                                                                          
         END AS QUARTER
, NULL AS MONTH
,ROUND(SUM(OvertimeHours),0) as TOTAL_OVERTIME 
from opsgroup.ic.ExpectedvsPaidHours
WHERE JobFamilyGroup in ('RECONDITIONING','POST_PRODUCTION')
AND Date >='2025-01-01'
AND Date < '2026-01-01'
GROUP BY LocationName,DATETRUNC(QUARTER,Date),JobFamilyGroup

UNION ALL

-- v3: 2026+ monthly data with dynamic Month/Quarter (SQL Server syntax)
Select LocationName as LOCATION,
DATETRUNC(QUARTER,Date) as DATE_QUARTER,
CASE 
 WHEN JobFamilyGroup like '%RECON%' Then 'Reconditioning'
 WHEN JobFamilyGroup like '%POST%' Then 'Post Production'
 ELSE NULL
 END
 AS JOB_FAMILY_GROUP
,CONCAT('Q', DATEPART(QUARTER, Date), '''', RIGHT(DATEPART(YEAR, Date), 2)) AS QUARTER
,FORMAT(Date, 'yyyy-MM') AS MONTH
,ROUND(SUM(OvertimeHours),0) as TOTAL_OVERTIME 
from opsgroup.ic.ExpectedvsPaidHours
WHERE JobFamilyGroup in ('RECONDITIONING','POST_PRODUCTION')
AND Date >= '2026-01-01'
GROUP BY LocationName,DATETRUNC(QUARTER,Date),JobFamilyGroup,FORMAT(Date, 'yyyy-MM')"""


overtime_sql = pd.read_sql_query(overtimequery, odbcconnops)

In [ ]:
overtime_sql.head()

In [ ]:
overtime_snow = overtime_sql.copy()

In [ ]:
overtime_snow.rename(columns={'LOCATION':'Location','JOB_FAMILY_GROUP':'JobFamilyGroup','QUARTER':'Quarter','TOTAL_OVERTIME':'Total_Overtime_hours_Quarter'},inplace=True)

In [ ]:
# v3: Add Month to overtime before merge
if 'Month' not in overtime_snow.columns:
    overtime_snow['Month'] = None
main_df=pd.merge(main_df,overtime_snow[['Location','JobFamilyGroup','Quarter','Month','Total_Overtime_hours_Quarter']],on=['Location','JobFamilyGroup','Quarter','Month'],how='left')

# Snowflake - Main DF

In [ ]:
# # Create Table
# db_snowflake_write.create_table('metrics',
#                   {
#                      'Location': ['VARCHAR(1000)'],
#                      'Score': ['INT'],
#                      'eSat Change': ['INT'], 
#                      'JobFamilyGroup': ['VARCHAR(1000)'],
#                      'Quarter': ['VARCHAR(1000)'], 
#                      'Injuries': ['INT'],
#                      'EmployeesPIPs_CARs': ['INT'],
#                      'CARs': ['INT'],
#                      'PIPs': ['INT'],
#                      'PIPs & CARs': ['INT'],
#                      'Terminations': ['INT'],
#                      'Quaterly_Turnover': ['INT'],
#                      'Turnover_Change':['INT'],
#                      'Total_Employees': ['INT'],
#                   })

In [ ]:
# main_df[main_df['Quarter']=='Q3\'25'][['Location','JobFamilyGroup','Score','Employee_Size_Peakon','Avg_emp']]

In [ ]:
db_snowflake_write.drop_table('metrics')
db_snowflake_write.dump('metrics',main_df)

In [ ]:
# Inserting the main df
# db_snowflake_write.insert('metrics',main_df,truncate=True)

In [ ]:
# db_snowflake_write.drop_table('main_df')